# Lab: Attention From Scratch

**Bluevision-ai Academy**
*The Attention Breakthrough: Query, Key, Value*

In lecture we watched the RNN bottleneck squeeze an entire sentence into one vector, then broke it open with a learned, dynamic weighting over every encoder state, and finally wrote that weighting down as a single formula: softmax(QK^T / sqrt(d_k))V. This lab builds that formula yourself, in code, and then asks it two questions the lecture only argued about in words: does it actually work on real language, and does it actually know anything about word order?

Each part is a hands-on build: you start from the problem, make the design choices, and interpret what your code actually produces. Five moves structure each part: **Problem** (what we have, what we want, why the obvious tool doesn't reach it) -> **Idea** (the reasoning move, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what just happened, and what it means). One idea per cell -- if a cell needs "and" to describe it, it's doing too much.

| Part | What you build | Builds on (lecture) |
|---|---|---|
| 1 -- From Scratch | Scaled dot-product attention as a raw formula, verified against a hand-built case where the right answer is known | "Scaled Dot-Product Attention" |
| 2 -- Real Translation | Cross-lingual attention on real English-French sentence pairs, with zero learned parameters | "Why Not the Embeddings Themselves?", "Query, Key, Value" |
| 3 -- Break It Deliberately | A precise, provable answer to whether attention knows word order | "Self-Attention", "The Problem That Remains" |
| Challenge | A deliberately crude patch, and an honest look at what it doesn't fix | "The Problem That Remains" (Session 5 preview) |
| Reflection | Five questions, answered in code comments | -- |

**Data:** real English-French sentence pairs pulled live from a public-domain literary translation corpus (`Helsinki-NLP/opus_books`, via Hugging Face `datasets`), and real pretrained multilingual contextual embeddings from `distilbert-base-multilingual-cased` (via Hugging Face `transformers`) -- no hand-typed toy sentences, no invented numbers. **Estimated time:** ~2 hours. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

The multilingual model download is the only slow step here (roughly a minute on a fresh runtime, cached after that). Everything else installs in seconds.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade torch transformers datasets plotly

import numpy as np
import torch
import plotly.graph_objects as go

torch.manual_seed(0)
np.random.seed(0)

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#7f8c9a"

## Part 1 -- Scaled Dot-Product Attention From Scratch

**Problem.** The "Scaled Dot-Product Attention" slide derived softmax(QK^T / sqrt(d_k))V one coloured term at a time, by hand, on paper. Every attention implementation you will ever import -- PyTorch's own, Hugging Face's, the ones the rest of this course leans on -- hides that exact formula behind a function call. The applied problem: you need this primitive as a real building block for the rest of this lab, and you cannot debug, extend, or trust a mechanism you have never implemented yourself.

**Ingredients.** Nothing external yet -- this part builds a generic primitive, not a mechanism tied to any one sentence. Part 2 is the first part that feeds it real data.

In [2]:
# A fixed embedding size for everything in this notebook, so shapes line up
# across parts without re-deriving them each time.
D_MODEL = 768  # matches the pretrained encoder Part 2 will use

**Idea -- write the formula, not a library call.** Implement exactly the four pieces the "Scaled Dot-Product Attention" slide walked through: the raw scores QK^T, the sqrt(d_k) rescaling, softmax, and the weighted sum with V. No `nn.MultiheadAttention`, no `torch.nn.functional.scaled_dot_product_attention` -- those are the black boxes this part exists to open. The lecture promised this fits in fewer than ten lines; hold it to that.

In [3]:
def attention(Q, K, V):
    """Q: (n_q, d_k)  K, V: (n_k, d_k)  ->  output: (n_q, d_v), weights: (n_q, n_k)"""
    d_k = Q.shape[-1]
    scores = Q @ K.T / d_k ** 0.5
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

**Idea -- trust, but verify.** Same discipline the k-means lab used on its own assign/update loop: before trusting this on real language, check it against a case small enough to reason about by hand. Build three keys that are mutually orthogonal (so there is no ambiguity about which one a query is "closest" to), give the query a huge dot product with exactly one of them, and check that almost all of the output weight lands there.

In [4]:
K_toy = torch.eye(3, 4)                                  # three orthogonal 4-d keys
V_toy = torch.tensor([[10., 0, 0, 0],
                       [0, 20., 0, 0],
                       [0, 0, 30., 0]])                     # a distinct, checkable value per key
Q_toy = torch.tensor([[0., 20., 0., 0.]])                   # built to align almost entirely with key 1

out_toy, w_toy = attention(Q_toy, K_toy, V_toy)

print("attention weights:", w_toy.numpy().round(4))
print("output:            ", out_toy.numpy().round(3))
print("weight on key 1:    %.4f" % w_toy[0, 1].item())

assert torch.isclose(w_toy.sum(), torch.tensor(1.0), atol=1e-5), "weights must sum to 1"
assert w_toy[0, 1] > 0.99, "the aligned key should take almost all the weight"
assert torch.allclose(out_toy[0], V_toy[1], atol=0.5), "output should land close to that key's value"
print("\ntoy check passed: attention() finds the aligned key and returns (almost) its value.")

attention weights: [[0.     0.9999 0.    ]]
output:             [[0.0000e+00 1.9998e+01 1.0000e-03 0.0000e+00]]
weight on key 1:    0.9999

toy check passed: attention() finds the aligned key and returns (almost) its value.


**Payoff.** The toy case passes on all three checks: the weights form a real probability distribution, the deliberately-aligned key takes over 99% of it, and the output nearly reproduces that key's value exactly. That confirms the implementation is doing what the formula says on a case where the right answer is known in advance -- the same "trust, but verify" step the k-means lab ran on its own inertia curve. The honest limitation: this is single-head, unbatched, and has no masking, all real machinery that a working transformer wraps around this exact computation (Session 5 territory). But the core arithmetic -- the thing every one of those extras eventually calls -- is exactly the six lines above, and they are now checked, not just trusted.

## Part 2 -- Testing It On Real Translation

**Problem.** The "Why Not the Embeddings Themselves?" slide argued, in words, that raw word vectors are not optimised for cross-word comparison, and that Bahdanau's model needs a *learned* scoring function to make that comparison meaningful. That is a claim about training, not a measurement. This part tests it: take real English-French sentence pairs -- not the lecture's own hand-illustrated "cat sat" example -- and see how far attention gets on real cross-lingual alignment with **zero learned attention parameters at all**.

**Ingredients.** A pretrained multilingual encoder (`distilbert-base-multilingual-cased`) supplies real contextual word vectors for both languages from a single shared model. A public-domain literary English-French translation corpus (`Helsinki-NLP/opus_books`, streamed live) supplies real sentence pairs -- filtered only for length and punctuation cleanliness, never hand-picked for a good outcome.

In [5]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME)
encoder.eval()

def word_embeddings(words):
    """One real contextual vector per word, mean-pooled over its subword pieces
    (a French word like "assis" often tokenises into two or three pieces)."""
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt")
    with torch.no_grad():
        hidden = encoder(**enc).last_hidden_state[0]      # (n_tokens, 768)
    word_ids = enc.word_ids()
    vecs = torch.zeros(len(words), hidden.shape[-1])
    counts = torch.zeros(len(words))
    for tok_i, wid in enumerate(word_ids):
        if wid is None:                                    # [CLS] / [SEP] / padding
            continue
        vecs[wid] += hidden[tok_i]
        counts[wid] += 1
    return vecs / counts.unsqueeze(1)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from datasets import load_dataset

books = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train", streaming=True)

def clean_pairs(stream, n, min_words=5, max_words=8):
    """Real sentence pairs, filtered only for length and punctuation cleanliness --
    never hand-picked for a good result."""
    found = []
    for ex in stream:
        en, fr = ex["translation"]["en"].strip(), ex["translation"]["fr"].strip()
        ew, fw = en.rstrip(".").split(), fr.rstrip(".").split()
        ok = (en.isascii() and en.endswith(".") and en[:1].isupper()
              and "'" not in en and '"' not in en
              and "," not in en and ";" not in en and "..." not in en
              and len(ew) == len(fw) and min_words <= len(ew) <= max_words)
        if ok:
            found.append((ew, fw))
        if len(found) >= n:
            break
    return found

pairs = clean_pairs(iter(books), n=4)
primary_en, primary_fr = pairs[0]
print(f"pulled {len(pairs)} real pairs; primary example:")
print(" ".join(primary_en))
print(" ".join(primary_fr))

pulled 4 real pairs; primary example:
The day had been deadly tedious
La journée avait été mortellement ennuyeuse


**Idea -- feed the embeddings straight into `attention()`, no learned projections.** Use each language's contextual vectors directly as Q (English) and K/V (French) -- literally the comparison the "Why Not the Embeddings Themselves?" slide said was not good enough, tested for real instead of only argued. The design choice this skips: Bahdanau's model *learns* a scoring function end to end on translation pairs; this uses the encoder exactly as it comes, pretrained on nothing but predicting masked words in ordinary text. Cheap -- no training run of our own -- but per the slide's own argument, we should expect a real gap.

In [7]:
Q_primary = word_embeddings(primary_en)
K_primary = word_embeddings(primary_fr)
_, weights_primary = attention(Q_primary, K_primary, K_primary)

fig = go.Figure(go.Heatmap(z=weights_primary.numpy(), x=primary_fr, y=primary_en,
                            colorscale=[[0, "#f4f4f4"], [1, GOLD]], zmin=0, zmax=1))
fig.update_layout(title="Real attention weights -- raw pretrained embeddings, zero learned parameters",
                   xaxis_title="French (keys)", yaxis_title="English (queries)",
                   width=650, height=480, yaxis=dict(autorange="reversed"))
fig.show()

argmax_fr = [primary_fr[i] for i in weights_primary.argmax(dim=1).tolist()]
for en_word, fr_word in zip(primary_en, argmax_fr):
    print(f"  {en_word:>14s}  ->  {fr_word}")

             The  ->  La
             day  ->  avait
             had  ->  avait
            been  ->  été
          deadly  ->  mortellement
         tedious  ->  ennuyeuse


**Payoff.** Five of six English words land on the semantically right French word with zero learned attention parameters: "The" -> "La", "had" -> "avait", "been" -> "été", "deadly" -> "mortellement", "tedious" -> "ennuyeuse". The one miss is real and instructive: "day" attends most strongly to "avait" rather than "journée" -- "avait" ("had") is a high-frequency auxiliary verb whose contextual vector apparently sits close to many different English words in this shared space, a real instance of the risk the "Why Not the Embeddings Themselves?" slide predicted: nothing in this encoder's own training ever asked it to be good at *this* comparison. The practical takeaway: raw pretrained contextual embeddings get you most of a working alignment for free, but not all of it -- the gap is real, not hypothetical.

**Idea -- check more than one sentence before trusting a single heatmap.** One heatmap could be a lucky (or unlucky) accident of that particular pair. Repeat the identical zero-training measurement across the other real pairs already pulled above, and score each English word by whether its highest-weighted French word sits in the same position -- an honest, simple heuristic for these near word-for-word literary pairs, not a claim about alignment in general.

In [8]:
total_words, total_hits = 0, 0
for ew, fw in pairs:
    Q, K = word_embeddings(ew), word_embeddings(fw)
    _, w = attention(Q, K, K)
    same_slot_hits = sum(1 for i, a in enumerate(w.argmax(dim=1).tolist()) if i == a)
    total_words += len(ew)
    total_hits += same_slot_hits
    print(f"{' '.join(ew):55s} {same_slot_hits}/{len(ew)}")

print(f"\naggregate: {total_hits}/{total_words} words landed on the same-position French word "
      f"({100 * total_hits / total_words:.0f}%)")

The day had been deadly tedious                         5/6
I know that Meaulnes has gone                           6/6
Generally he set down three                             3/5
A moment later everything disappears behind the hedge   8/8

aggregate: 22/25 words landed on the same-position French word (88%)


**Payoff.** Across four independent real sentence pairs, 22 of 25 words (88%) land on the same-position French word using nothing but the pretrained encoder's own contextual vectors -- consistent with the first heatmap, not a fluke of one lucky example (the range across pairs runs from 3/5 to a perfect 8/8). That is a strong result for a mechanism that has never seen a single translation pair during training. It also sharpens the "Why Not the Embeddings Themselves?" slide's claim: a genuinely useful cross-lingual comparison space already exists inside a model trained only to predict masked words, so Bahdanau's learned scoring function is not starting from nothing -- it is refining a space that is already most of the way there. Practical takeaway: before reaching for a learned projection, it is worth measuring, on real data, how far the representations you already have will carry you -- here, further than the slide's argument alone would suggest.

## Part 3 -- Break It Deliberately: Does Attention Know Word Order?

**Problem.** The narration promised a scramble test. Sharpened into a real question: does the `attention()` function itself carry any notion of sequence order, or does it only ever see an unordered set of vectors? This can be answered precisely, not just eyeballed on a shifted heatmap.

**Ingredients.** Reuse `Q_primary` and `K_primary` from Part 2 -- no new data, no new model call.

**Idea -- permute the matrix, not the sentence.** Re-running the *scrambled English sentence* back through the multilingual encoder would also change every word's contextual vector, since that encoder has its own built-in sense of position -- that would confound two different effects (the encoder's context-sensitivity and attention's own behaviour) in one test. To isolate the attention step alone, permute the **rows of the already-computed embedding matrix** directly and feed that reordered matrix into the exact same `attention()` from Part 1. If attention has no notion of order, permuting which row is which should permute the output rows and nothing else.

In [9]:
perm = torch.randperm(len(primary_en))
scrambled_en = [primary_en[i] for i in perm.tolist()]
print("original: ", " ".join(primary_en))
print("scrambled:", " ".join(scrambled_en), " (rows reordered, not re-encoded)")

Q_scrambled = Q_primary[perm]
_, weights_scrambled = attention(Q_scrambled, K_primary, K_primary)

# Row i of the scrambled run should be EXACTLY row perm[i] of the original run.
predicted = weights_primary[perm]
max_diff = (weights_scrambled - predicted).abs().max().item()
print(f"\nmax difference between scrambled row i and original row perm[i]: {max_diff:.2e}")
assert max_diff < 1e-5, "attention should be exactly permutation-equivariant over its queries"
print("confirmed: scrambling the query order only reorders the output rows -- nothing else changes.")

original:  The day had been deadly tedious
scrambled: had tedious been The day deadly  (rows reordered, not re-encoded)

max difference between scrambled row i and original row perm[i]: 0.00e+00
confirmed: scrambling the query order only reorders the output rows -- nothing else changes.


**Payoff.** The measured difference is 0.00e+00 -- bit-for-bit identical in this run, not merely small. That is the expected result, not a coincidence: `attention()` computes weights for row i as softmax(Q_i @ K^T / sqrt(d_k)), using only Q_i's own values and the fixed K, V matrices. Nowhere in that computation does the function reference i itself or any other row of Q. Relabelling which row is "first" or "third" cannot change a calculation that never looked at row position in the first place, so permuting the queries is guaranteed to permute the outputs identically, exactly, every time -- not just usually. Practical consequence: "the dog bit the man" and "the man bit the dog" would receive the exact same per-word attention behaviour, word for word, from this mechanism alone -- two sentences with opposite meanings, computationally indistinguishable to attention by itself.

## Challenge -- Break It Deliberately: Can You Patch It Yourself?

**Problem.** Part 3 proved attention cannot see word order at all -- not approximately, exactly. Before Session 5 shows you the real fix: try the cheapest possible patch yourself, apply it, and see honestly what it does and does not buy you.

**Ingredients.** Reuse the same `perm` from Part 3 -- nothing new to load.

**Idea -- the crudest possible position signal.** Add one extra number to every word's vector based on **its slot in the sequence** (0, 1, 2, ...), not its identity -- the simplest signal that could possibly carry "where in the list am I". The design choice, stated plainly: this is deliberately not a well-engineered scheme. Trade-off: instant to add, but arbitrary in scale, and it grows without bound as sentences get longer -- exactly the kind of rough edge a real fix has to smooth out, which is precisely what Session 5 exists to do properly.

In [10]:
def tag_with_slot(Q, scale=0.5):
    """Adds scale * (slot index) to every dimension -- crude on purpose."""
    slots = torch.arange(Q.shape[0]).unsqueeze(1).float()
    return Q + scale * slots

Q_primary_tagged = tag_with_slot(Q_primary)
Q_scrambled_tagged = tag_with_slot(Q_primary[perm])   # tagged by the NEW slot, after permuting

_, weights_primary_tagged = attention(Q_primary_tagged, K_primary, K_primary)
_, weights_scrambled_tagged = attention(Q_scrambled_tagged, K_primary, K_primary)

predicted_tagged = weights_primary_tagged[perm]
max_diff_tagged = (weights_scrambled_tagged - predicted_tagged).abs().max().item()
print(f"max difference, no position tag (Part 3):  {max_diff:.2e}")
print(f"max difference, WITH position tag:          {max_diff_tagged:.2e}")

max difference, no position tag (Part 3):  0.00e+00
max difference, WITH position tag:          2.54e-02


**Payoff.** With the slot tag, the same test now measures 2.54e-02 instead of 0.00e+00 -- adding one number per word, keyed only to *where it sits*, is enough to break the exact equivalence Part 3 proved. Word order now measurably changes the attention weights. But look at how it was done: `scale * slot_index` grows without bound (a fifty-word sentence would carry a tag fifty times the size of a one-word one), the scale `0.5` was picked by hand with no evidence it is the right size relative to a 768-dimensional embedding, and nothing about this ramp was learned from data the way every other number in this notebook's mechanism was.

**Your answer (2-3 sentences).** Before Session 5: in your own words, what is the one piece of information attention structurally cannot see, and what would a *well-designed* fix need to guarantee that this slot-index patch does not (think about what should stay true if the same word sits at position 3 of a 6-word sentence versus position 3 of a 60-word sentence)?

*(write your answer here)*

## Reflection

**Problem.** You now own the full mechanism this session introduced: the formula, a working real-data test of the "why not raw embeddings" question, and a precise proof of what attention cannot see on its own. The point of this last cell is not a right answer -- it is stating, in your own words, the reasoning each part was built on.

In [11]:
# 1. Part 1's toy check confirmed attention() is CORRECTLY IMPLEMENTED (it finds
#    the aligned key, the weights sum to 1). Part 2 then ran that same, correct
#    function on real language and got an imperfect result. What does putting
#    these two findings side by side tell you about the difference between an
#    attention mechanism being CORRECT and an attention mechanism being GOOD?
answer_1 = """
"""

# 2. Part 2's aggregate same-position hit rate used real pretrained embeddings
#    and exactly zero learned attention parameters. If you then trained W_Q, W_K,
#    W_V end to end on a real translation objective (Bahdanau's actual approach),
#    what would you expect to happen to the words this notebook's raw version
#    got wrong, and why would training help specifically there?
answer_2 = """
"""

# 3. Part 3's permutation test came out EXACTLY equal (down to floating-point
#    noise), not just close. Walk through, in your own words, why the attention
#    formula guarantees this exactly -- what does each query's score depend on,
#    and what does it never depend on?
answer_3 = """
"""

# 4. The Challenge's slot-index patch broke Part 3's exact equivalence with one
#    added line. Name the single biggest reason you would NOT ship that patch
#    in a real model, and describe -- without looking ahead -- one property a
#    well-designed positional signal would need that this one does not have.
answer_4 = """
"""

# 5. Across all four parts, if a friend asked why attention needed a whole new
#    formula instead of just comparing the word embeddings you already had from
#    Session 2, which ONE number from this notebook would you show them first,
#    and why that one over the others?
answer_5 = """
"""

---
*The Attention Breakthrough: Query, Key, Value · Bluevision-ai Academy*